In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/anrf-aise-hack-pha

In [2]:
# =============================================================================
# CELL 1 - Imports, environment, paths, and core configuration
# =============================================================================

import gc
import json
import math
import os
import random
import time
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import io as sio
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(42)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()

print(f"Device  : {device}")
print(f"GPUs    : {n_gpus}")
for idx in range(n_gpus):
    props = torch.cuda.get_device_properties(idx)
    print(f"  GPU {idx}: {props.name} | VRAM={props.total_memory / 1e9:.1f} GB")

COMP_ROOT = (
    "/kaggle/input/competitions/"
    "anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2"
)
RAW_PATH = os.path.join(COMP_ROOT, "raw")
TEST_INPUT_LOC = os.path.join(COMP_ROOT, "test_in")
MIN_MAX_FILE = os.path.join(COMP_ROOT, "stats", "feat_min_max.mat")

CACHE_ROOT = "/kaggle/temp/aisehack_v2"
STACKED_DIR = os.path.join(CACHE_ROOT, "stacked")
CHECKPOINT_DIR = "/kaggle/working/checkpoints_v2"
LOG_DIR = "/kaggle/working/logs_v2"
OUTPUT_PATH = "/kaggle/working/preds.npy"
ZSTATS_PATH = os.path.join("/kaggle/working", "zstats_v2.npz")

for folder in [CACHE_ROOT, STACKED_DIR, CHECKPOINT_DIR, LOG_DIR]:
    os.makedirs(folder, exist_ok=True)

TIME_INPUT = 10
TIME_OUT = 16
HORIZON = TIME_INPUT + TIME_OUT
S1, S2 = 140, 124
N_TEST = 218

BASE_MET_VARS = ["cpm25", "q2", "t2", "u10", "swdown", "pblh", "v10", "psfc", "rain"]
EMI_VARS = ["PM25", "NH3", "SO2", "NOx", "NMVOC_e", "NMVOC_finn", "bio"]
BASE_VARS = BASE_MET_VARS + EMI_VARS
DERIVED_VARS = [
    "wind_speed",
    "wind_div",
    "wind_vort",
    "vent_coeff",
    "inv_vent",
    "rel_humidity",
    "emission_sum",
    "emission_vent_ratio",
]
ALL_VARS = BASE_VARS + DERIVED_VARS
PM25_IDX = ALL_VARS.index("cpm25")
N_CHANNELS = len(ALL_VARS)

ALL_MONTHS = ["APRIL_16", "JULY_16", "OCT_16", "DEC_16"]

# Recommended tuning setup:
# - use an entire month for validation
# - after choosing a good epoch count, switch RUN_MODE to "submit"
RUN_MODE = "tune"  # "tune" or "submit"
VAL_MONTH = "DEC_16"
if RUN_MODE == "tune":
    TRAIN_MONTHS = [m for m in ALL_MONTHS if m != VAL_MONTH]
    EVAL_MONTHS = [VAL_MONTH]
else:
    TRAIN_MONTHS = list(ALL_MONTHS)
    EVAL_MONTHS = []

BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 2
TUNE_EPOCHS = 24
SUBMIT_EPOCHS = 12  # set this to the best epoch count you observe in tune mode
EPOCHS = TUNE_EPOCHS if RUN_MODE == "tune" else SUBMIT_EPOCHS
LR = 2.0e-4
WEIGHT_DECAY = 1.0e-4
GRAD_CLIP = 1.0
NUM_WORKERS = 4
STRIDE_TRAIN = 1
STRIDE_VAL = 1
PATIENCE = 6

BASE_WIDTH = 32
BOTTLENECK_WIDTH = 128
SPECTRAL_MODES = 12
DROPOUT = 0.10

supports_amp = (
    torch.cuda.is_available()
    and torch.cuda.get_device_capability(0)[0] >= 7
)
USE_AMP = bool(supports_amp)

BEST_PATH = os.path.join(CHECKPOINT_DIR, "best_v2.pt")
LAST_PATH = os.path.join(CHECKPOINT_DIR, "last_v2.pt")
LOG_PATH = os.path.join(LOG_DIR, "train_log_v2.json")

print(
    f"\nRUN_MODE={RUN_MODE} | train_months={TRAIN_MONTHS} | val_months={EVAL_MONTHS}\n"
    f"BATCH_SIZE={BATCH_SIZE} | ACCUM={GRAD_ACCUM_STEPS} | EPOCHS={EPOCHS} | "
    f"WIDTH={BASE_WIDTH} | BOTTLENECK={BOTTLENECK_WIDTH} | AMP={USE_AMP}\n"
    f"TOTAL_CHANNELS={N_CHANNELS}"
     )

Device  : cuda
GPUs    : 2
  GPU 0: Tesla T4 | VRAM=15.6 GB
  GPU 1: Tesla T4 | VRAM=15.6 GB

RUN_MODE=tune | train_months=['APRIL_16', 'JULY_16', 'OCT_16'] | val_months=['DEC_16']
BATCH_SIZE=4 | ACCUM=2 | EPOCHS=24 | WIDTH=32 | BOTTLENECK=128 | AMP=True
TOTAL_CHANNELS=24


In [3]:
# =============================================================================
# CELL 2 - Physics features, z-score stats, and cached stacked arrays
# =============================================================================


def load_month_raw(month: str, root: str = RAW_PATH) -> Dict[str, np.ndarray]:
    return {
        feat: np.load(os.path.join(root, month, f"{feat}.npy")).astype(np.float32)
        for feat in BASE_VARS
    }


def load_test_raw(root: str = TEST_INPUT_LOC) -> Dict[str, np.ndarray]:
    return {
        feat: np.load(os.path.join(root, f"{feat}.npy")).astype(np.float32)
        for feat in BASE_VARS
    }


def wind_derivatives(u: np.ndarray, v: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    dudx = np.gradient(u, axis=2)
    dvdy = np.gradient(v, axis=1)
    dvdx = np.gradient(v, axis=2)
    dudy = np.gradient(u, axis=1)
    div = dudx + dvdy
    vort = dvdx - dudy
    return div.astype(np.float32), vort.astype(np.float32)


def relative_humidity_from_qtp(
    q2: np.ndarray, t2: np.ndarray, psfc: np.ndarray
) -> np.ndarray:
    q = np.clip(q2, 1e-8, None).astype(np.float32)
    t_c = (t2 - 273.15).astype(np.float32)
    p = np.clip(psfc, 100.0, None).astype(np.float32)
    vapor_pressure = (q * p) / np.clip(0.622 + 0.378 * q, 1e-6, None)
    sat_pressure = 611.2 * np.exp((17.67 * t_c) / np.clip(t_c + 243.5, 1.0, None))
    rh = vapor_pressure / np.clip(sat_pressure, 1.0, None)
    return np.clip(rh, 0.0, 1.5).astype(np.float32)


def compute_derived_features(raw: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    u10 = raw["u10"]
    v10 = raw["v10"]
    pblh = np.clip(raw["pblh"], 1.0, None)
    wind_speed = np.sqrt(u10 * u10 + v10 * v10).astype(np.float32)
    wind_div, wind_vort = wind_derivatives(u10, v10)
    vent_coeff = (wind_speed * pblh).astype(np.float32)
    inv_vent = (1.0 / np.sqrt(vent_coeff + 1.0)).astype(np.float32)
    rel_humidity = relative_humidity_from_qtp(raw["q2"], raw["t2"], raw["psfc"])
    emission_sum = np.zeros_like(raw["PM25"], dtype=np.float32)
    for feat in EMI_VARS:
        emission_sum += np.clip(raw[feat], 0.0, None).astype(np.float32)
    emission_vent_ratio = (emission_sum / np.sqrt(vent_coeff + 1.0)).astype(np.float32)

    return {
        "wind_speed": wind_speed,
        "wind_div": wind_div,
        "wind_vort": wind_vort,
        "vent_coeff": vent_coeff,
        "inv_vent": inv_vent,
        "rel_humidity": rel_humidity,
        "emission_sum": emission_sum.astype(np.float32),
        "emission_vent_ratio": emission_vent_ratio.astype(np.float32),
    }


def iter_feature_arrays(raw: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    derived = compute_derived_features(raw)
    out = {}
    for feat in BASE_VARS:
        out[feat] = raw[feat].astype(np.float32)
    for feat in DERIVED_VARS:
        out[feat] = derived[feat].astype(np.float32)
    return out


def compute_zstats(train_months: List[str], force: bool = False) -> Dict[str, np.ndarray]:
    if os.path.exists(ZSTATS_PATH) and not force:
        stats_npz = np.load(ZSTATS_PATH)
        stats = {
            "mean": stats_npz["mean"].astype(np.float32),
            "std": stats_npz["std"].astype(np.float32),
            "names": stats_npz["names"],
        }
        print(f"Loaded z-score stats from {ZSTATS_PATH}")
        return stats

    total_sum = np.zeros(N_CHANNELS, dtype=np.float64)
    total_sq = np.zeros(N_CHANNELS, dtype=np.float64)
    total_count = 0

    print("Computing z-score stats from training months...")
    for month in train_months:
        raw = load_month_raw(month)
        feats = iter_feature_arrays(raw)
        n = feats[ALL_VARS[0]].size
        total_count += n
        for idx, feat in enumerate(ALL_VARS):
            arr = feats[feat].astype(np.float64)
            total_sum[idx] += arr.sum()
            total_sq[idx] += np.square(arr).sum()
        del raw, feats
        gc.collect()

    mean = total_sum / max(total_count, 1)
    var = total_sq / max(total_count, 1) - mean ** 2
    std = np.sqrt(np.clip(var, 1e-8, None))

    stats = {
        "mean": mean.astype(np.float32),
        "std": std.astype(np.float32),
        "names": np.asarray(ALL_VARS),
    }
    np.savez_compressed(ZSTATS_PATH, **stats)
    print(f"Saved z-score stats to {ZSTATS_PATH}")
    return stats


def stack_and_normalize(
    raw: Dict[str, np.ndarray],
    stats: Dict[str, np.ndarray],
) -> np.ndarray:
    feats = iter_feature_arrays(raw)
    tensors = []
    for idx, feat in enumerate(ALL_VARS):
        arr = feats[feat]
        arr = (arr - stats["mean"][idx]) / stats["std"][idx]
        arr = np.clip(arr, -8.0, 8.0).astype(np.float32)
        tensors.append(arr[..., None])
    return np.concatenate(tensors, axis=-1).astype(np.float16)


def build_cached_month(month: str, stats: Dict[str, np.ndarray], force: bool = False) -> str:
    out_path = os.path.join(STACKED_DIR, f"{month}_stacked.npy")
    if os.path.exists(out_path) and not force:
        return out_path
    raw = load_month_raw(month)
    stacked = stack_and_normalize(raw, stats)
    np.save(out_path, stacked)
    del raw, stacked
    gc.collect()
    return out_path


def build_cached_test(stats: Dict[str, np.ndarray], force: bool = False) -> str:
    out_path = os.path.join(STACKED_DIR, "test_stacked.npy")
    if os.path.exists(out_path) and not force:
        return out_path
    raw = load_test_raw()
    stacked = stack_and_normalize(raw, stats)
    np.save(out_path, stacked)
    del raw, stacked
    gc.collect()
    return out_path


zstats = compute_zstats(TRAIN_MONTHS, force=False)
train_month_files = [build_cached_month(month, zstats, force=False) for month in TRAIN_MONTHS]
val_month_files = [build_cached_month(month, zstats, force=False) for month in EVAL_MONTHS]
test_file = build_cached_test(zstats, force=False)

print("Cached month files:")
for path in train_month_files + val_month_files + [test_file]:
    print(f"  {path}")

Computing z-score stats from training months...
Saved z-score stats to /kaggle/working/zstats_v2.npz
Cached month files:
  /kaggle/temp/aisehack_v2/stacked/APRIL_16_stacked.npy
  /kaggle/temp/aisehack_v2/stacked/JULY_16_stacked.npy
  /kaggle/temp/aisehack_v2/stacked/OCT_16_stacked.npy
  /kaggle/temp/aisehack_v2/stacked/DEC_16_stacked.npy
  /kaggle/temp/aisehack_v2/stacked/test_stacked.npy


In [4]:
# =============================================================================
# CELL 3 - Datasets and dataloaders
# =============================================================================


@dataclass
class SampleIndex:
    file_path: str
    start: int


class WindowedMonthDataset(Dataset):
    def __init__(self, file_paths: List[str], stride: int = 1):
        self.file_paths = list(file_paths)
        self.stride = stride
        self.mmaps = {path: np.load(path, mmap_mode="r") for path in self.file_paths}
        self.indices: List[SampleIndex] = []
        for path in self.file_paths:
            total_steps = self.mmaps[path].shape[0]
            max_start = total_steps - HORIZON
            for start in range(0, max_start + 1, self.stride):
                self.indices.append(SampleIndex(path, start))
        print(
            f"{self.__class__.__name__}: files={len(self.file_paths)} "
            f"samples={len(self.indices)} stride={self.stride}"
        )

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        entry = self.indices[idx]
        arr = self.mmaps[entry.file_path][entry.start : entry.start + HORIZON].astype(np.float32)

        x = torch.from_numpy(arr[:TIME_INPUT]).permute(0, 3, 1, 2).contiguous()
        future = torch.from_numpy(arr[TIME_INPUT:, :, :, PM25_IDX]).contiguous()
        last_pm25 = torch.from_numpy(arr[TIME_INPUT - 1, :, :, PM25_IDX]).contiguous()

        pm25_mean = float(zstats["mean"][PM25_IDX])
        pm25_std = float(zstats["std"][PM25_IDX])
        future_phys = future * pm25_std + pm25_mean
        last_pm25_phys = last_pm25 * pm25_std + pm25_mean

        return {
            "x": x,
            "y_norm": future,
            "last_norm": last_pm25,
            "y_phys": future_phys,
            "last_phys": last_pm25_phys,
        }


class TestDataset(Dataset):
    def __init__(self, file_path: str):
        self.arr = np.load(file_path, mmap_mode="r")
        self.n = self.arr.shape[0]
        print(f"TestDataset: N={self.n} expected={N_TEST}")
        assert self.n == N_TEST, f"Expected {N_TEST}, got {self.n}"

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, idx: int):
        x = self.arr[idx, :TIME_INPUT].astype(np.float32)
        return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous()


def make_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
        persistent_workers=NUM_WORKERS > 0,
    )


train_ds = WindowedMonthDataset(train_month_files, stride=STRIDE_TRAIN)
train_loader = make_loader(train_ds, shuffle=True)

if val_month_files:
    val_ds = WindowedMonthDataset(val_month_files, stride=STRIDE_VAL)
    val_loader = make_loader(val_ds, shuffle=False)
else:
    val_ds = None
    val_loader = None

test_ds = TestDataset(test_file)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=NUM_WORKERS > 0,
)

WindowedMonthDataset: files=3 samples=2118 stride=1
WindowedMonthDataset: files=1 samples=714 stride=1
TestDataset: N=218 expected=218


In [5]:
# =============================================================================
# CELL 4 - Hybrid ConvLSTM U-Net with spectral bottleneck
# =============================================================================


class ConvGNAct(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 3, s: int = 1, p: int = 1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.GroupNorm(num_groups=min(8, out_ch), num_channels=out_ch),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ResidualConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.0):
        super().__init__()
        self.conv1 = ConvGNAct(in_ch, out_ch)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=min(8, out_ch), num_channels=out_ch),
        )
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.skip(x)
        x = self.conv1(x)
        x = self.dropout(x)
        x = self.conv2(x)
        return F.gelu(x + residual)


class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, kernel_size: int = 3):
        super().__init__()
        padding = kernel_size // 2
        self.hidden_dim = hidden_dim
        self.gates = nn.Conv2d(
            input_dim + hidden_dim,
            4 * hidden_dim,
            kernel_size=kernel_size,
            padding=padding,
        )

    def forward(
        self,
        x: torch.Tensor,
        h: torch.Tensor,
        c: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        combined = torch.cat([x, h], dim=1)
        gates = self.gates(combined)
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c

    def init_hidden(
        self, batch: int, spatial: Tuple[int, int], device_: torch.device, dtype: torch.dtype
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        h, w = spatial
        shape = (batch, self.hidden_dim, h, w)
        return (
            torch.zeros(shape, device=device_, dtype=dtype),
            torch.zeros(shape, device=device_, dtype=dtype),
        )


class SpectralResidualBlock(nn.Module):
    def __init__(self, channels: int, modes: int):
        super().__init__()
        self.channels = channels
        self.modes = modes
        scale = 1.0 / math.sqrt(channels)
        self.wr = nn.Parameter(scale * torch.randn(channels, channels, modes, modes))
        self.wi = nn.Parameter(scale * torch.randn(channels, channels, modes, modes))
        self.local = nn.Sequential(
            ResidualConvBlock(channels, channels),
            ResidualConvBlock(channels, channels),
        )
        self.norm = nn.GroupNorm(min(8, channels), channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        b, c, h, w = x.shape
        x_ft = torch.fft.rfft2(x.float(), dim=(-2, -1))
        out_ft = torch.zeros(
            b, c, h, (w // 2) + 1, dtype=torch.cfloat, device=x.device
        )
        wr = self.wr
        wi = self.wi
        weights = torch.complex(wr, wi)
        modes_h = min(self.modes, h)
        modes_w = min(self.modes, (w // 2) + 1)
        out_ft[:, :, :modes_h, :modes_w] = torch.einsum(
            "bixy,ioxy->boxy",
            x_ft[:, :, :modes_h, :modes_w],
            weights[:, :, :modes_h, :modes_w],
        )
        spectral = torch.fft.irfft2(out_ft, s=(h, w)).to(x.dtype)
        x = residual + spectral + self.local(x)
        return F.gelu(self.norm(x))


class EncoderStep(nn.Module):
    def __init__(self, in_ch: int, base_width: int):
        super().__init__()
        self.stem = ResidualConvBlock(in_ch, base_width)
        self.down1 = nn.Sequential(nn.AvgPool2d(2), ResidualConvBlock(base_width, base_width * 2))
        self.down2 = nn.Sequential(
            nn.AvgPool2d(2), ResidualConvBlock(base_width * 2, base_width * 4)
        )
        self.down3 = nn.Sequential(
            nn.AvgPool2d(2), ResidualConvBlock(base_width * 4, BOTTLENECK_WIDTH)
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        x1 = self.stem(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        return x1, x2, x3, x4


class UpBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.conv = ResidualConvBlock(in_ch + skip_ch, out_ch, dropout=DROPOUT)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class PollutionNet(nn.Module):
    def __init__(self, in_ch: int, base_width: int = 32, modes: int = 12):
        super().__init__()
        self.encoder = EncoderStep(in_ch, base_width)
        self.temporal1 = ConvLSTMCell(base_width, base_width)
        self.temporal2 = ConvLSTMCell(base_width * 2, base_width * 2)
        self.temporal3 = ConvLSTMCell(base_width * 4, base_width * 4)
        self.temporal4 = ConvLSTMCell(BOTTLENECK_WIDTH, BOTTLENECK_WIDTH)

        self.bottleneck = nn.Sequential(
            SpectralResidualBlock(BOTTLENECK_WIDTH, modes=modes),
            SpectralResidualBlock(BOTTLENECK_WIDTH, modes=modes),
        )

        self.up3 = UpBlock(BOTTLENECK_WIDTH, base_width * 4, base_width * 4)
        self.up2 = UpBlock(base_width * 4, base_width * 2, base_width * 2)
        self.up1 = UpBlock(base_width * 2, base_width, base_width)
        self.refine = nn.Sequential(
            ResidualConvBlock(base_width, base_width, dropout=DROPOUT),
            ResidualConvBlock(base_width, base_width, dropout=DROPOUT),
        )
        self.delta_head = nn.Conv2d(base_width, TIME_OUT, kernel_size=1)
        self.episode_head = nn.Conv2d(base_width, TIME_OUT, kernel_size=1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # x: (B, T, C, H, W)
        b, t, _, h, w = x.shape
        dtype = x.dtype
        h2_size, w2_size = h // 2, w // 2
        h3_size, w3_size = h // 4, w // 4
        h4_size, w4_size = h // 8, w // 8

        h1, c1 = self.temporal1.init_hidden(b, (h, w), x.device, dtype)
        h2, c2 = self.temporal2.init_hidden(b, (h2_size, w2_size), x.device, dtype)
        h3, c3 = self.temporal3.init_hidden(b, (h3_size, w3_size), x.device, dtype)
        h4, c4 = self.temporal4.init_hidden(b, (h4_size, w4_size), x.device, dtype)

        for step in range(t):
            x1, x2, x3, x4 = self.encoder(x[:, step])
            h1, c1 = self.temporal1(x1, h1, c1)
            h2, c2 = self.temporal2(x2, h2, c2)
            h3, c3 = self.temporal3(x3, h3, c3)
            h4, c4 = self.temporal4(x4, h4, c4)

        x = self.bottleneck(h4)
        x = self.up3(x, h3)
        x = self.up2(x, h2)
        x = self.up1(x, h1)
        x = self.refine(x)
        delta = self.delta_head(x)
        episode_logits = self.episode_head(x)
        return delta, episode_logits


_model_check = PollutionNet(in_ch=N_CHANNELS, base_width=BASE_WIDTH, modes=SPECTRAL_MODES)
_dummy = torch.randn(2, TIME_INPUT, N_CHANNELS, S1, S2)
_delta, _ep = _model_check(_dummy)
assert _delta.shape == (2, TIME_OUT, S1, S2)
assert _ep.shape == (2, TIME_OUT, S1, S2)
print(f"Model parameters: {sum(p.numel() for p in _model_check.parameters()):,}")
del _model_check, _dummy, _delta, _ep
gc.collect()

Model parameters: 14,663,008


0

In [6]:
# =============================================================================
# CELL 5 - Competition-aware loss, training, and validation
# =============================================================================


def smape(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    denom = 0.5 * (pred.abs() + target.abs()) + eps
    return (pred - target).abs() / denom


def detect_episode_mask(target_phys: torch.Tensor, thresh: float = 1.0) -> torch.Tensor:
    # target_phys: (B, T, H, W)
    mu = target_phys.mean(dim=(-2, -1), keepdim=True)
    std = target_phys.std(dim=(-2, -1), keepdim=True)
    return target_phys > (mu + thresh * std)


def horizon_weights(device_: torch.device) -> torch.Tensor:
    weights = torch.linspace(1.0, 1.35, TIME_OUT, device=device_)
    return weights.view(1, TIME_OUT, 1, 1)


class CompetitionLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(
        self,
        pred_norm_delta: torch.Tensor,
        episode_logits: torch.Tensor,
        batch: Dict[str, torch.Tensor],
    ) -> Tuple[torch.Tensor, Dict[str, float], torch.Tensor]:
        last_norm = batch["last_norm"].unsqueeze(1)
        target_norm = batch["y_norm"]
        last_phys = batch["last_phys"].unsqueeze(1)
        target_phys = batch["y_phys"]

        pred_norm = pred_norm_delta + last_norm
        pm25_mean = float(zstats["mean"][PM25_IDX])
        pm25_std = float(zstats["std"][PM25_IDX])
        pred_phys = pred_norm * pm25_std + pm25_mean
        pred_phys = torch.clamp(pred_phys, min=0.0)

        hw = horizon_weights(pred_phys.device)
        global_smape = (smape(pred_phys, target_phys) * hw).mean()

        with torch.no_grad():
            ep_mask = detect_episode_mask(target_phys, thresh=1.0)
        ep_mask_f = ep_mask.float()
        ep_count = ep_mask_f.sum().clamp(min=1.0)

        episode_smape = (smape(pred_phys, target_phys) * ep_mask_f * hw).sum() / ep_count

        corr_terms = []
        for t in range(TIME_OUT):
            m = ep_mask[:, t]
            if m.sum() < 4:
                continue
            p = pred_phys[:, t][m]
            y = target_phys[:, t][m]
            p = p - p.mean()
            y = y - y.mean()
            corr = (p * y).sum() / (torch.norm(p) * torch.norm(y)).clamp(min=1e-6)
            corr_terms.append(corr)

        episode_corr = (
            torch.stack(corr_terms).mean()
            if corr_terms
            else torch.tensor(0.0, device=pred_phys.device)
        )
        corr_loss = 1.0 - episode_corr

        aux_hotspot = self.bce(episode_logits, ep_mask_f)
        robust_delta = F.smooth_l1_loss(pred_norm, target_norm)

        total = (
            0.90 * global_smape
            + 1.35 * episode_smape
            + 0.80 * corr_loss
            + 0.25 * aux_hotspot
            + 0.30 * robust_delta
        )

        metrics = {
            "loss": float(total.detach().item()),
            "g_smape": float(global_smape.detach().item()),
            "ep_smape": float(episode_smape.detach().item()),
            "ep_corr": float(episode_corr.detach().item()),
            "hotspot_bce": float(aux_hotspot.detach().item()),
        }
        return total, metrics, pred_phys


def competition_like_score(g_smape: float, ep_smape: float, ep_corr: float) -> float:
    norm_g = 1.0 - (g_smape / 2.0)
    norm_ep_s = 1.0 - (ep_smape / 2.0)
    norm_ep_c = (ep_corr + 1.0) / 2.0
    norm_g = max(0.0, min(1.0, norm_g))
    norm_ep_s = max(0.0, min(1.0, norm_ep_s))
    norm_ep_c = max(0.0, min(1.0, norm_ep_c))
    return 0.34 * norm_g + 0.33 * norm_ep_s + 0.33 * norm_ep_c


def move_batch(batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def mean_dict(rows: List[Dict[str, float]]) -> Dict[str, float]:
    if not rows:
        return {}
    keys = rows[0].keys()
    return {k: float(np.mean([row[k] for row in rows])) for k in keys}


print("\nBuilding model...")
base_model = PollutionNet(
    in_ch=N_CHANNELS,
    base_width=BASE_WIDTH,
    modes=SPECTRAL_MODES,
).to(device)
if n_gpus > 1:
    model = nn.DataParallel(base_model)
    print(f"DataParallel enabled across {n_gpus} GPUs")
else:
    model = base_model

criterion = CompetitionLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS, 1))
scaler = GradScaler(enabled=USE_AMP)

best_score = -1.0
start_epoch = 0
history: List[Dict[str, float]] = []

if os.path.exists(BEST_PATH) and RUN_MODE == "tune":
    ckpt = torch.load(BEST_PATH, map_location=device)
    state = {k.replace("module.", ""): v for k, v in ckpt["model_state_dict"].items()}
    base_model.load_state_dict(state, strict=True)
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    best_score = float(ckpt["best_score"])
    start_epoch = int(ckpt["epoch"]) + 1
    history = ckpt.get("history", [])
    print(f"Resumed from epoch {start_epoch} with best_score={best_score:.5f}")

print(
    f"\n{'Ep':>3} {'Time':>6} {'LR':>10} {'TrainLoss':>10} "
    f"{'ValLoss':>10} {'G-SMAPE':>9} {'EP-SMAPE':>10} {'EP-Corr':>8} {'Score':>8}"
)
print("-" * 88)

epochs_without_improve = 0
for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_rows = []

    for step, batch in enumerate(train_loader, start=1):
        batch = move_batch(batch)
        with autocast(enabled=USE_AMP):
            pred_delta, episode_logits = model(batch["x"])
            loss, metrics, _ = criterion(pred_delta, episode_logits, batch)
            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        train_rows.append(metrics)

    train_mean = mean_dict(train_rows)
    elapsed = time.time() - t0
    lr_now = optimizer.param_groups[0]["lr"]

    if val_loader is not None:
        model.eval()
        val_rows = []
        with torch.no_grad():
            for batch in val_loader:
                batch = move_batch(batch)
                with autocast(enabled=USE_AMP):
                    pred_delta, episode_logits = model(batch["x"])
                    _, metrics, _ = criterion(pred_delta, episode_logits, batch)
                val_rows.append(metrics)
        val_mean = mean_dict(val_rows)
        score = competition_like_score(
            val_mean["g_smape"], val_mean["ep_smape"], val_mean["ep_corr"]
        )
        improved = score > best_score
        if improved:
            best_score = score
            epochs_without_improve = 0
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": base_model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_score": best_score,
                    "history": history,
                    "config": {
                        "run_mode": RUN_MODE,
                        "train_months": TRAIN_MONTHS,
                        "val_month": VAL_MONTH,
                        "channels": ALL_VARS,
                    },
                },
                BEST_PATH,
            )
        else:
            epochs_without_improve += 1

        print(
            f"{epoch:3d} {elapsed:5.0f}s {lr_now:10.2e} {train_mean['loss']:10.4f} "
            f"{val_mean['loss']:10.4f} {val_mean['g_smape']:9.4f} "
            f"{val_mean['ep_smape']:10.4f} {val_mean['ep_corr']:8.4f} {score:8.4f}"
        )
        if improved:
            print(f"  best checkpoint saved -> {BEST_PATH}")
    else:
        val_mean = {}
        score = float("nan")
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": base_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_score": best_score,
                "history": history,
            },
            BEST_PATH,
        )
        print(
            f"{epoch:3d} {elapsed:5.0f}s {lr_now:10.2e} {train_mean['loss']:10.4f} "
            f"{0.0:10.4f} {0.0:9.4f} {0.0:10.4f} {0.0:8.4f} {0.0:8.4f}"
        )

    history.append(
        {
            "epoch": epoch,
            "elapsed_sec": round(elapsed, 2),
            "lr": lr_now,
            "train_loss": train_mean.get("loss", float("nan")),
            "train_g_smape": train_mean.get("g_smape", float("nan")),
            "train_ep_smape": train_mean.get("ep_smape", float("nan")),
            "train_ep_corr": train_mean.get("ep_corr", float("nan")),
            "val_loss": val_mean.get("loss", float("nan")),
            "val_g_smape": val_mean.get("g_smape", float("nan")),
            "val_ep_smape": val_mean.get("ep_smape", float("nan")),
            "val_ep_corr": val_mean.get("ep_corr", float("nan")),
            "score": score,
        }
    )
    with open(LOG_PATH, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": base_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_score": best_score,
            "history": history,
        },
        LAST_PATH,
    )

    scheduler.step()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if val_loader is not None and epochs_without_improve >= PATIENCE:
        print(f"Early stopping triggered after {epoch + 1} epochs")
        break


Building model...
DataParallel enabled across 2 GPUs

 Ep   Time         LR  TrainLoss    ValLoss   G-SMAPE   EP-SMAPE  EP-Corr    Score
----------------------------------------------------------------------------------------
  0   328s   2.00e-04     1.1396     1.2139    0.4631     0.2532   0.7019   0.8303
  best checkpoint saved -> /kaggle/working/checkpoints_v2/best_v2.pt
  1   270s   1.99e-04     0.8765     1.0674    0.4203     0.2332   0.7328   0.8460
  best checkpoint saved -> /kaggle/working/checkpoints_v2/best_v2.pt
  2   270s   1.97e-04     0.7132     1.0740    0.4602     0.2458   0.7324   0.8370
  3   270s   1.92e-04     0.6101     1.0765    0.4437     0.2692   0.7367   0.8367
  4   270s   1.87e-04     0.5444     1.0785    0.4254     0.2798   0.7303   0.8370
  5   270s   1.79e-04     0.5014     1.0671    0.4200     0.2779   0.7331   0.8387
  6   270s   1.71e-04     0.4666     1.0911    0.4111     0.2974   0.7273   0.8360
  7   271s   1.61e-04     0.4389     1.1668    0.4859 

In [7]:
# =============================================================================
# CELL 6 - Inference and submission verification
# =============================================================================


print("\nLoading best checkpoint for inference...")
ckpt = torch.load(BEST_PATH, map_location=device)
state = {k.replace("module.", ""): v for k, v in ckpt["model_state_dict"].items()}
base_model.load_state_dict(state, strict=True)
base_model.eval()

pm25_mean = float(zstats["mean"][PM25_IDX])
pm25_std = float(zstats["std"][PM25_IDX])
preds = np.zeros((N_TEST, S1, S2, TIME_OUT), dtype=np.float32)

cursor = 0
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        batch = batch.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP):
            pred_delta, _ = base_model(batch)
        last_norm = batch[:, TIME_INPUT - 1, PM25_IDX]
        pred_norm = pred_delta + last_norm.unsqueeze(1)
        pred_phys = pred_norm * pm25_std + pm25_mean
        pred_phys = torch.clamp(pred_phys, min=0.0)
        pred_phys = pred_phys.permute(0, 2, 3, 1).contiguous().cpu().numpy()
        bs = pred_phys.shape[0]
        preds[cursor : cursor + bs] = pred_phys
        cursor += bs

np.save(OUTPUT_PATH, preds.astype(np.float32))

print("\nSubmission summary")
print("=" * 60)
print(f"Checkpoint : {BEST_PATH}")
print(f"Output     : {OUTPUT_PATH}")
print(f"Shape      : {preds.shape}")
print(f"Dtype      : {preds.dtype}")
print(f"Min / Max  : {preds.min():.3f} / {preds.max():.3f}")
print(f"Mean       : {preds.mean():.3f}")
print(f"NaNs       : {np.isnan(preds).sum()}")
print(f"Infs       : {np.isinf(preds).sum()}")
print(f"Negatives  : {(preds < 0).sum()}")
print("=" * 60)

assert preds.shape == (N_TEST, S1, S2, TIME_OUT)
assert preds.dtype == np.float32
assert not np.isnan(preds).any()
assert not np.isinf(preds).any()
assert not (preds < 0).any()

print("\nReady for Kaggle submission.")


Loading best checkpoint for inference...


Inference:   0%|          | 0/55 [00:00<?, ?it/s]


Submission summary
Checkpoint : /kaggle/working/checkpoints_v2/best_v2.pt
Output     : /kaggle/working/preds.npy
Shape      : (218, 140, 124, 16)
Dtype      : float32
Min / Max  : 0.000 / 417.324
Mean       : 36.433
NaNs       : 0
Infs       : 0
Negatives  : 0

Ready for Kaggle submission.
